# 📘 Notebook 16: Planning and Tree Search

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module E: Beyond a Single Agent · Notebook 1 of 3 in this module · (16 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Distinguish **learning from experience** from **planning with a model**, and say what each needs
- Implement **Dyna-Q**, which learns a model from experience and uses it for extra, imagined updates
- Compute perfect play for a small game by **exhaustive search**, and explain why that fails for larger games
- Describe the four steps of **Monte Carlo tree search (MCTS)** and implement it
- Measure how the strength of MCTS grows with the **number of simulations**
- Explain how AlphaZero combines tree search with neural networks and self-play

> **Prerequisites:** Notebook 01 (the UCB rule), Notebook 03 (planning with a known model) and Notebook 05 (Q-learning).
>
> 🔭 **Why this notebook matters later:** every agent since Notebook 04 was model-free: it reacted, quickly, with whatever its table or network had stored. Planning is the other half of intelligent behaviour, thinking ahead before acting. The strongest game-playing programs combine the two, and the next notebook supplies the missing piece, an opponent.

> 📦 **Libraries used in this notebook.** Only NumPy, matplotlib and **imageio**. There are no neural networks here, and every cell runs in seconds, except two matches of Connect Four that take up to a minute.

In [ ]:
%pip install -q imageio imageio-ffmpeg

In [ ]:
import base64
import math
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import imageio
from IPython.display import HTML, display


def show_video(frames, fps=2, width=280):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. Reacting and thinking ahead

### Intuition first
A chess beginner sees a piece that can be captured and captures it. A stronger player stops and thinks: "if I take, she takes back, then I can move there, and then..." The first player **reacts**. The second **plans**: they try moves out in their head, on an imagined board, before touching a real piece.

Imagining requires knowing what a move will lead to. That knowledge is a **model** of the environment.

### Formal definition

| | Model-free learning | Planning |
|---|---|---|
| Needs | experience: real steps in the environment | a model: what each action leads to |
| Produces | values or a policy, stored for instant use | a decision, or improved values, by computation |
| Met in | Notebooks 04 to 15 | Notebook 03 (value iteration) |
| Cost | many real steps | much thinking time |

Notebook 03 planned with a model that was **given**. This notebook covers two further situations: the model has to be **learned** (Section 2), and the model is known but the game is far too large to plan exhaustively (Sections 3 to 5).

## 2. Dyna-Q: learning a model and using it

### Intuition first
You get lost once in a new town and finally find the station. Walking home, you go over the route in your mind: "from the fountain, left was a dead end, right led to the bridge..." Each real step of that walk was experienced once, but you **replay** it many times in thought, and next day you know the way far better than one walk alone would explain.

### Formal definition
**Dyna-Q** (Richard Sutton, 1990) is Q-learning with two additions:

1. After every real step $(S, A, R, S')$ the agent records it in a **model**: "in state $S$, action $A$ gave reward $R$ and led to $S'$".
2. It then performs $n$ **planning steps**: it picks a remembered state-action pair at random, asks the model what happened there, and applies the ordinary Q-learning update to that imagined transition.

With $n = 0$ this is plain Q-learning.

The environment is a small maze. Every step gives a reward of 0, and reaching the goal gives +1, so the reward is rare: the situation in which tabular Q-learning is slowest, because the information has to creep back from the goal one step per visit.

In [ ]:
class Maze:
    """The maze of Sutton and Barto's Dyna example: reach G from S. Every step gives 0, reaching G gives +1."""
    LAYOUT = ["       #G",
              "  #    # ",
              "S #    # ",
              "  #      ",
              "     #   ",
              "         "]
    MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]

    def __init__(self):
        self.rows, self.cols = len(self.LAYOUT), len(self.LAYOUT[0])
        self.n_states, self.n_actions = self.rows * self.cols, 4
        self.start = next(r * self.cols + c for r in range(self.rows) for c in range(self.cols) if self.LAYOUT[r][c] == "S")

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        row, col = divmod(self.state, self.cols)
        r, c = row + self.MOVES[action][0], col + self.MOVES[action][1]
        if 0 <= r < self.rows and 0 <= c < self.cols and self.LAYOUT[r][c] != "#":
            self.state = r * self.cols + c
        done = self.LAYOUT[self.state // self.cols][self.state % self.cols] == "G"
        return self.state, float(done), done


def dyna_q(env, episodes, planning_steps, alpha=0.1, gamma=0.95, epsilon=0.1, seed=0):
    """Tabular Dyna-Q. Returns the Q-table and the number of real steps of each episode."""
    rng = random.Random(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    model = {}                                              # (state, action) -> (reward, next state, done)
    steps_per_episode = []
    for episode in range(episodes):
        state, steps, done = env.reset(), 0, False
        while not done:
            if rng.random() < epsilon:
                action = rng.randrange(4)
            else:
                best = np.flatnonzero(Q[state] == Q[state].max())
                action = int(best[rng.randrange(len(best))])
            next_state, reward, done = env.step(action)
            steps += 1

            # learn from the real step, as in Notebook 05
            Q[state, action] += alpha * (reward + gamma * (0.0 if done else Q[next_state].max()) - Q[state, action])
            # remember it
            model[(state, action)] = (reward, next_state, done)
            # learn from imagined steps
            known = list(model)
            for i in range(planning_steps):
                s, a = known[rng.randrange(len(known))]
                r, s2, d = model[(s, a)]
                Q[s, a] += alpha * (r + gamma * (0.0 if d else Q[s2].max()) - Q[s, a])

            state = next_state
        steps_per_episode.append(steps)
    return Q, steps_per_episode


maze = Maze()
print("\n".join(Maze.LAYOUT).replace(" ", "."))

plt.figure(figsize=(8, 4))
for planning_steps in (0, 5, 50):
    curves = [dyna_q(maze, 30, planning_steps, seed=seed)[1] for seed in range(10)]
    average = np.mean(curves, axis=0)
    plt.plot(range(1, 31), average, label=f"{planning_steps} planning steps")
    print(f"{planning_steps:>2} planning steps: {average[1]:5.0f} real steps in episode 2, {average[4]:4.0f} in episode 5, "
          f"{int(average.sum()):>5} real steps in total")
plt.ylim(0, 400)
plt.title("Dyna-Q in the maze (average of 10 runs)")
plt.xlabel("episode"); plt.ylabel("real steps needed to reach the goal"); plt.grid(True); plt.legend()
plt.show()

The first episode is a blind search for every agent, since nobody knows where the goal is. The difference shows from the second episode on. Plain Q-learning has learned the value of exactly one step, the last one, and wanders for hundreds of steps again. The agent with 50 planning steps has replayed its first episode thousands of times in imagination, and the value of the goal has already spread back along the whole route.

All three agents took the same kind of real steps. The planners simply **learned more from each of them**. When real experience is expensive, such as a robot's time or a patient's treatment, and computation is cheap, that is the trade to make.

> 🧠 This should sound familiar. The replay buffer of Notebook 09 also reuses each real step many times. A replay buffer is a simple model: one that can only repeat what has actually happened. A learned model can do more, such as answering questions about actions that were never tried in a given state, and it can also be **wrong**, which a recording cannot.

## 3. Games against an opponent, and perfect play

For the rest of the notebook we change the kind of problem. In a board game the model is **known exactly**: the rules say what every move leads to. What makes the game hard is the opponent, and the number of positions.

We describe a game by four functions:

| Function | Meaning |
|---|---|
| `start()` | the initial state |
| `moves(state)` | the legal moves |
| `play(state, move)` | the state after a move (the state itself is never changed) |
| `winner(state)` | 1 or 2 if that player has won, 0 for a draw, `None` if the game is still going on |

A state is a pair: the board, and the player whose turn it is.

In [ ]:
class TicTacToe:
    """State: (board as a tuple of 9 cells, player to move). Cells are 0 (empty), 1 or 2."""
    LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]
    def start(self): return ((0,) * 9, 1)
    def moves(self, state): return [i for i, cell in enumerate(state[0]) if cell == 0]
    def play(self, state, move):
        board, player = state; board = list(board); board[move] = player
        return (tuple(board), 3 - player)
    def winner(self, state):
        """1 or 2 if that player has won, 0 for a draw, None if the game is not over."""
        board = state[0]
        for a, b, c in self.LINES:
            if board[a] and board[a] == board[b] == board[c]: return board[a]
        return 0 if all(board) else None


def show_board(state):
    symbols = {0: ".", 1: "X", 2: "O"}
    board = state[0]
    for row in range(3):
        print(" ".join(symbols[board[3 * row + col]] for col in range(3)))


game = TicTacToe()
state = game.start()
for move in (4, 0, 8):
    state = game.play(state, move)
show_board(state)
print("legal moves:", game.moves(state), "  winner:", game.winner(state))

### Exhaustive search
Tic-tac-toe is small enough to solve completely. The value of a position for the player to move is +1 if they can force a win, 0 if the best they can reach is a draw, and −1 if they lose against correct play. It follows from the values of the positions one move later: **my best move is the one that leaves my opponent in the worst position**. That is the Bellman optimality equation of Notebook 03, with a minus sign because the players alternate. The method is called **minimax**.

In [ ]:
def solve(game):
    """Value and best move of every reachable position, by exhaustive search."""
    table = {}

    def value(state):                                   # from the point of view of the player to move
        if state in table:
            return table[state][0]
        result = game.winner(state)
        if result is not None:
            v = 0 if result == 0 else (1 if result == state[1] else -1)
            table[state] = (v, None)
            return v
        best_value, best_move = -2, None
        for move in game.moves(state):
            v = -value(game.play(state, move))          # what is good for my opponent is bad for me
            if v > best_value:
                best_value, best_move = v, move
        table[state] = (best_value, best_move)
        return best_value

    value(game.start())
    return table


start = time.time()
perfect_table = solve(game)
print(f"positions examined: {len(perfect_table):,}   time: {time.time() - start:.2f} seconds")
print("value of the empty board for the first player:", perfect_table[game.start()][0], "(0 means a draw)")

A few thousand positions, a fraction of a second, and the game is **solved**: with correct play by both sides, tic-tac-toe is a draw. The table also gives a perfect player, which we will use as a measuring stick.

### Why this does not scale
Connect Four, on a board of 6 rows and 7 columns, has about 4.5 trillion positions. Chess and Go have more than there are atoms in the observable universe. No table will ever hold them. A program for such games can only look at a tiny part of the tree of possibilities, and the whole art is in choosing **which** part.

## 4. Monte Carlo tree search

### Intuition first
You have to choose a restaurant in a street you do not know, and you have time to look briefly into a few. You do not inspect each one thoroughly, and you do not pick at random. You glance at all of them, go back for a closer look at the ones that seemed promising, and keep half an eye on the ones you have barely seen, in case you misjudged them.

That is the exploration and exploitation trade-off of Notebook 01. **Monte Carlo tree search (MCTS)** treats the choice of a move as a multi-armed bandit, at every position of the tree it builds.

And how does it judge a position without a table of values? In the simplest possible way: it plays the game to the end with **random moves** and sees who wins. One random game says almost nothing. The average of hundreds says a surprising amount.

### Formal definition
MCTS grows a tree of positions, starting from the current one. Each **simulation** has four steps:

1. **Selection.** From the root, walk down the tree. At each node choose the child with the best UCB score, $\dfrac{w}{n} + c\sqrt{\dfrac{\ln N}{n}}$, where $w$ and $n$ are the wins and visits of the child and $N$ the visits of the parent.
2. **Expansion.** On reaching a node with a move that has not been tried, add one new child for it.
3. **Simulation** (also called a rollout). From the new node, play random moves to the end of the game.
4. **Backpropagation.** Walk back up to the root, adding one visit to every node on the path and crediting the result to the player who made the move into that node.

After all simulations the program plays the move at the root that was **visited most often**.

In [ ]:
class Node:
    def __init__(self, state, parent=None, move=None):
        self.state, self.parent, self.move = state, parent, move
        self.children = []
        self.untried = None                  # moves from here that have no child yet
        self.visits = 0
        self.wins = 0.0                      # counted for the player who made the move INTO this node


def mcts(game, state, simulations, c=1.4, rng=random):
    """Returns the chosen move and the root of the search tree."""
    root = Node(state)
    root.untried = game.moves(state)
    for i in range(simulations):
        node = root
        # 1. selection
        while not node.untried and node.children:
            log_parent = math.log(node.visits)
            node = max(node.children, key=lambda child: child.wins / child.visits
                       + c * math.sqrt(log_parent / child.visits))
        # 2. expansion
        if node.untried:
            move = node.untried.pop(rng.randrange(len(node.untried)))
            child = Node(game.play(node.state, move), node, move)
            child.untried = game.moves(child.state) if game.winner(child.state) is None else []
            node.children.append(child)
            node = child
        # 3. simulation: random moves to the end of the game
        state_now = node.state
        result = game.winner(state_now)
        while result is None:
            moves = game.moves(state_now)
            state_now = game.play(state_now, moves[rng.randrange(len(moves))])
            result = game.winner(state_now)
        # 4. backpropagation
        while node is not None:
            node.visits += 1
            mover = 3 - node.state[1]                    # the player who moved into this node
            node.wins += 1.0 if result == mover else 0.5 if result == 0 else 0.0
            node = node.parent
    return max(root.children, key=lambda child: child.visits).move, root

Note what the function does **not** contain: no table, no network, no knowledge of tic-tac-toe beyond the four functions of the game. It is given a position and a budget of simulations, and it thinks.

A first test. Player X has two in a row and threatens to win at square 2. It is O's turn.

In [ ]:
rng = random.Random(0)
state = ((1, 1, 0,
          0, 2, 0,
          0, 0, 0), 2)
show_board(state)

move, root = mcts(game, state, 500, rng=rng)
print("\nvisits per move:", {child.move: child.visits for child in sorted(root.children, key=lambda ch: ch.move)})
print("chosen move:", move)

Most of the simulations went into the one move that blocks the threat. Nobody told the program about threats. Every other move simply kept losing in the simulations that followed it, and the UCB rule stopped spending time there.

### How good is it?
We let MCTS play against a random player and against the perfect player from Section 3, alternating who moves first.

In [ ]:
def play_game(game, player_1, player_2):
    """A player is a function from a state to a move. Returns the winner (1, 2 or 0)."""
    players = {1: player_1, 2: player_2}
    state = game.start()
    while game.winner(state) is None:
        state = game.play(state, players[state[1]](state))
    return game.winner(state)


def match(game, player_a, player_b, games):
    """Play `games` games, alternating who starts. Returns wins of A, wins of B, draws."""
    score = {"A": 0, "B": 0, "draw": 0}
    for i in range(games):
        if i % 2 == 0:
            result = play_game(game, player_a, player_b)
            score[{1: "A", 2: "B", 0: "draw"}[result]] += 1
        else:
            result = play_game(game, player_b, player_a)
            score[{1: "B", 2: "A", 0: "draw"}[result]] += 1
    return score


def random_player(game):
    return lambda state: rng.choice(game.moves(state))


def mcts_player(game, simulations, c=1.4):
    return lambda state: mcts(game, state, simulations, c, rng)[0]


def perfect_player(state):
    return perfect_table[state][1]


print("MCTS (200 simulations) against a random player, 40 games:")
print("  ", match(game, mcts_player(game, 200), random_player(game), 40), " (A is MCTS)")

print("\nMCTS against the perfect player, 20 games each:")
for simulations in (10, 50, 200, 1000):
    print(f"   {simulations:>4} simulations per move:", match(game, mcts_player(game, simulations), perfect_player, 20))

Against the perfect player nobody can win, so the best possible result is a draw in every game. With a handful of simulations MCTS loses often. As the budget grows the losses disappear, and with enough simulations it draws every game: it **plays perfectly** without ever having stored a single position. Its strength is not learned. It is bought with thinking time, afresh at every move.

## 5. Connect Four

Now a game that cannot be solved by a table. Two players drop discs into seven columns, and the first to get four in a row, horizontally, vertically or diagonally, wins. Only the game class is new. `mcts` is used as it is.

In [ ]:
class ConnectFour:
    """State: (tuple of 7 columns, each a tuple of discs from the bottom up, player to move)."""
    ROWS, COLS = 6, 7
    def start(self): return (((),) * self.COLS, 1)
    def moves(self, state): return [c for c in range(self.COLS) if len(state[0][c]) < self.ROWS]
    def play(self, state, move):
        columns, player = state; columns = list(columns); columns[move] = columns[move] + (player,)
        return (tuple(columns), 3 - player)
    def winner(self, state):
        columns = state[0]
        grid = [[columns[c][r] if r < len(columns[c]) else 0 for c in range(self.COLS)] for r in range(self.ROWS)]
        for r in range(self.ROWS):
            for c in range(self.COLS):
                p = grid[r][c]
                if not p: continue
                for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                    rr, cc = r + 3 * dr, c + 3 * dc
                    if 0 <= rr < self.ROWS and 0 <= cc < self.COLS and all(grid[r + k * dr][c + k * dc] == p for k in range(4)):
                        return p
        return 0 if all(len(col) == self.ROWS for col in columns) else None


def draw_board(state, cell=40):
    """The board as a picture: yellow for player 1, red for player 2."""
    colours = {0: (245, 245, 245), 1: (240, 200, 40), 2: (210, 60, 60)}
    image = np.zeros((6 * cell, 7 * cell, 3), dtype=np.uint8)
    image[:] = (40, 80, 170)
    yy, xx = np.mgrid[0:cell, 0:cell]
    disc = (yy - cell / 2) ** 2 + (xx - cell / 2) ** 2 < (cell * 0.4) ** 2
    for col in range(7):
        for row in range(6):
            player = state[0][col][row] if row < len(state[0][col]) else 0
            top = (5 - row) * cell
            image[top:top + cell, col * cell:(col + 1) * cell][disc] = colours[player]
    return image


connect4 = ConnectFour()
state = connect4.start()
for move in (3, 3, 4, 2, 5):
    state = connect4.play(state, move)

plt.figure(figsize=(3.5, 3))
plt.imshow(draw_board(state))
plt.axis("off")
plt.show()
print("legal moves:", connect4.moves(state), "  player to move:", state[1])

Yellow (player 1) has three in a row on the bottom. The left end of the row is already blocked by a red disc, so yellow threatens to win in the last column on the right. It is red's turn. What does the search think?

In [ ]:
move, root = mcts(connect4, state, 2000, rng=rng)
for child in sorted(root.children, key=lambda ch: ch.move):
    print(f"column {child.move}: {child.visits:>5} visits, red won or drew {100 * child.wins / child.visits:3.0f}% of the simulations through it")
print("chosen column:", move)

The search puts most of its simulations into the last column, the only move that stops yellow from winning at once. In every other column red's chances are poor, because the simulations through them usually end with yellow completing the row. Reading a position like this is what a human player does, and here it emerges from random games and a counting rule.

Now two matches. In the first, MCTS with 200 simulations per move plays a random player. In the second, MCTS with 300 simulations plays MCTS with 30. **Each takes up to a minute.**

In [ ]:
start = time.time()
print("MCTS (200) against random, 10 games:  ", match(connect4, mcts_player(connect4, 200), random_player(connect4), 10))
print("MCTS (300) against MCTS (30), 10 games:", match(connect4, mcts_player(connect4, 300), mcts_player(connect4, 30), 10))
print(f"({time.time() - start:.0f} seconds)   A is the first-named player")

More thinking time means stronger play: the same program beats itself when one copy is given ten times the budget. That is a dial no model-free agent has. A trained DQN is exactly as good as its network, however long you let it "think" about a move.

In [ ]:
state = connect4.start()
frames = [draw_board(state)]
players = {1: mcts_player(connect4, 400), 2: mcts_player(connect4, 400)}
while connect4.winner(state) is None:
    state = connect4.play(state, players[state[1]](state))
    frames.append(draw_board(state))

result = connect4.winner(state)
print("result:", {1: "yellow wins", 2: "red wins", 0: "draw"}[result], f"after {len(frames) - 1} moves")
show_video(frames + [frames[-1]] * 3)

> ⚠️ **Common pitfalls**
>
> - **Crediting a result to the wrong player.** A node's statistics belong to the player who made the move **into** it. Getting this wrong makes the search choose the moves that are best for the opponent.
> - **Choosing the final move by win rate.** A move tried twice and won twice has a rate of 100% and means nothing. The number of visits is the reliable signal.
> - **Judging MCTS by a few games.** The results above are from ten or twenty games. Like everything in this course they scatter.
> - **Forgetting what random rollouts cannot see.** A position that is won only by one precise sequence of moves looks poor in random play, because random play almost never finds the sequence.

## 6. From tree search to AlphaZero

Pure MCTS has two weaknesses, and both come from its ignorance.

- **Random rollouts are a crude judge.** In a game such as Go, a random game says very little about who is really ahead.
- **It treats all moves alike at first.** In a position with 250 legal moves it has to try each of them before the statistics begin to guide it.

Now recall what Modules C and D built: networks that estimate the **value** of a state, and networks that give a **policy**, a probability for each action. They are exactly what the search lacks.

- A **value network** can judge a position directly. It replaces the random rollout.
- A **policy network** can say which moves look promising before any of them has been tried. It tells the search where to look first.

That combination is the idea behind DeepMind's **AlphaGo**, which defeated the world-class Go player Lee Sedol by four games to one in 2016. Its first version learned in part from games played by humans. Its successors, **AlphaGo Zero** and **AlphaZero** (2017), used none. They start from random play and repeat one loop:

1. Play games against **yourself**, choosing each move with MCTS guided by the current networks.
2. Train the policy network to predict the moves the search chose, and the value network to predict who won.
3. The better networks make the search stronger, which produces better games to learn from.

The search improves the networks, and the networks improve the search. With that loop and nothing but the rules, AlphaZero reached superhuman strength at chess, shogi and Go.

Two ingredients of the loop are still missing from this course. One is the tree search, which you have just written. The other is in step 1: learning from games against **yourself**. That is the subject of the next notebook.

---
## ✏️ Exercises

The exercises use `Maze`, `dyna_q`, `TicTacToe`, `perfect_table`, `mcts`, `match`, the player functions, `ConnectFour` and `draw_board`.

### Exercise 1 (How much planning?)
Run Dyna-Q in the maze for 30 episodes with 0, 1, 5, 20 and 100 planning steps, averaged over 10 seeds, and print for each the total number of **real** steps and the total number of **updates** to the Q-table (each real step is one update, and brings `planning_steps` more). What is being traded for what?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for planning_steps in (0, 1, 5, 20, 100):
    real = np.mean([sum(dyna_q(maze, 30, planning_steps, seed=seed)[1]) for seed in range(10)])
    print(f"{planning_steps:>3} planning steps: {real:6.0f} real steps,  {real * (1 + planning_steps):9.0f} updates in total")
```

*More planning means fewer real steps and more computation. The saving in real steps is large at first and then levels off. Beyond about 20 planning steps there is no further gain here, and the remaining differences are within the scatter of ten runs. The first episode must be explored in reality whatever happens, and no amount of thinking replaces experience that the agent does not have.*
</details>

### Exercise 2 (A position by hand)
In the tic-tac-toe position below it is X's turn. Work out on paper whether X can force a win, and with which move. Then look the position up in `perfect_table`.

In [ ]:
state = ((1, 0, 0,
          0, 2, 0,
          0, 0, 1), 2)
state = game.play(state, 2)        # O plays the top-right corner
show_board(state)
# Your analysis, then: print(perfect_table[state])


<details><summary>💡 Show solution</summary>

```python
state = ((1, 0, 0,
          0, 2, 0,
          0, 0, 1), 2)
state = game.play(state, 2)
value, move = perfect_table[state]
print("value for X:", value, "  best move:", move)
show_board(game.play(state, move))
```

*O has just made a threat along the diagonal from the top right to the bottom left, so X must first block at square 6. That block creates two threats for X at once, along the bottom row and down the left column, and O can only stop one of them. A value of +1 means exactly this: a win that the opponent cannot prevent.*
</details>

### Exercise 3 (The exploration constant)
`mcts` takes the constant `c` of the UCB rule. Let MCTS with 200 simulations play 20 games against the perfect tic-tac-toe player for `c` = 0, 0.5, 1.4 and 5, and print the results. Explain what goes wrong at each extreme.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for c in (0.0, 0.5, 1.4, 5.0):
    print(f"c = {c:<4}:", match(game, mcts_player(game, 200, c=c), perfect_player, 20), " (A is MCTS)")
```

*With c = 0 the search is greedy: it keeps returning to whichever move happened to win its first random game and hardly examines the others, like the greedy bandit agent of Notebook 01. It is clearly the worst setting. Among the other values the differences are no larger than the scatter of 20 games. A very large c spreads the simulations almost evenly over all moves. In a game as small as tic-tac-toe that does little harm, because 200 simulations cover everything anyway. In a large game it would leave no line examined deeply enough to see what follows from it.*
</details>

### Exercise 4 (Thinking time and strength)
For tic-tac-toe, measure the share of games MCTS does **not lose** against the perfect player for 5, 20, 50, 100, 300 and 1,000 simulations per move (30 games each), and plot it with a logarithmic horizontal axis.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
budgets = [5, 20, 50, 100, 300, 1000]
not_lost = []
for simulations in budgets:
    score = match(game, mcts_player(game, simulations), perfect_player, 30)
    not_lost.append(100 * (score["A"] + score["draw"]) / 30)
    print(f"{simulations:>4} simulations: MCTS avoided defeat in {not_lost[-1]:3.0f}% of games")

plt.figure(figsize=(7, 4))
plt.plot(budgets, not_lost, marker="o")
plt.xscale("log")
plt.title("MCTS against a perfect tic-tac-toe player")
plt.xlabel("simulations per move (logarithmic scale)"); plt.ylabel("% of games not lost"); plt.grid(True)
plt.show()
```

*The curve rises steeply and then flattens as it approaches 100%. Each doubling of the budget helps less than the one before, and beyond the point of perfect play it would help not at all. For Connect Four the same curve keeps rising over a far longer range, and for Go it has no visible end.*
</details>

### Exercise 5 (Smarter rollouts (a little harder))
Random rollouts overlook the obvious: a player with three in a row does not take the win. Write a rollout rule `sensible_move(game, state)` that plays a winning move if one exists and a random move otherwise. Copy `mcts` to `mcts_sensible`, replacing only the choice of move in step 3, and let it play 10 games of Connect Four against plain MCTS, both with 100 simulations per move. Time the match, and compare the time with the 20 games of Section 5. Is the smarter rollout worth its cost?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def sensible_move(game, state):
    moves = game.moves(state)
    for move in moves:
        if game.winner(game.play(state, move)) == state[1]:      # this move wins at once
            return move
    return moves[rng.randrange(len(moves))]


def mcts_sensible(game, state, simulations, c=1.4):
    root = Node(state)
    root.untried = game.moves(state)
    for i in range(simulations):
        node = root
        while not node.untried and node.children:
            log_parent = math.log(node.visits)
            node = max(node.children, key=lambda child: child.wins / child.visits
                       + c * math.sqrt(log_parent / child.visits))
        if node.untried:
            move = node.untried.pop(rng.randrange(len(node.untried)))
            child = Node(game.play(node.state, move), node, move)
            child.untried = game.moves(child.state) if game.winner(child.state) is None else []
            node.children.append(child)
            node = child
        state_now = node.state
        result = game.winner(state_now)
        while result is None:
            state_now = game.play(state_now, sensible_move(game, state_now))     # the only changed line
            result = game.winner(state_now)
        while node is not None:
            node.visits += 1
            mover = 3 - node.state[1]
            node.wins += 1.0 if result == mover else 0.5 if result == 0 else 0.0
            node = node.parent
    return max(root.children, key=lambda child: child.visits).move


start = time.time()
score = match(connect4, lambda state: mcts_sensible(connect4, state, 100), mcts_player(connect4, 100), 10)
print(score, f" (A uses sensible rollouts)   {time.time() - start:.0f} seconds")
```

*In our runs the match was close to even: 4 to 6 over these 10 games, and 10 to 8 over 20. Taking an immediate win is too small an improvement to show against the noise, and it makes every simulation several times slower, since each move of a rollout now tries all seven columns first. A stronger rule, which also **blocks** the opponent's immediate win, won 13 to 5 in our test and was about three times slower again. Better rollouts are not free, and whether better or more simulations pay off depends on the budget. AlphaZero goes to the extreme of this trade: it replaces the rollout by one evaluation of a large neural network.*
</details>

## 🔑 Key takeaways

- **Planning** uses a model to try actions in imagination. Model-free learning needs real experience and no model.
- **Dyna-Q** learns a model from real steps and replays imagined ones: far fewer real steps, more computation.
- A small game can be **solved exhaustively** with minimax. Real games have far too many positions.
- **MCTS** builds a tree selectively: UCB chooses where to look, random games estimate who is winning, and the most visited move is played.
- Its strength grows with the **number of simulations**: more thinking time, better play, with no learning at all.
- **AlphaZero** guides the search with a policy network, replaces rollouts with a value network, and trains both by **self-play**.

---
**Next:** *Notebook 17: Opponents and Self-Play*, where the other player learns too.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*